In [ ]:
import polars as pl
import numpy as np
import os

# Load PONSOL dataset
df = pl.read_csv(
    "../data/ponsol.csv",
    has_header=True,
)

# Select relevant columns and drop rows with missing target values
df = df.select(["sequence", "mutations", "solubility_change"]).drop_nulls(["solubility_change"])

df

In [ ]:
# Extract mutation details into separate columns (e.g., "T76I" -> from: T, pos: 76, to: I)
df = df.with_columns([
    pl.col("mutations").str.extract(r'([A-Za-z])(\d+)([A-Za-z])', 1).alias("mutation_from"),
    pl.col("mutations").str.extract(r'([A-Za-z])(\d+)([A-Za-z])', 2).cast(pl.Int64).alias("mutation_pos"),
    pl.col("mutations").str.extract(r'([A-Za-z])(\d+)([A-Za-z])', 3).alias("mutation_to"),
    pl.col("sequence").str.len_chars().alias("seq_len"),
])

# Verify amino acid at mutation position and generate mutated sequence
df = df.with_columns([
    pl.col("sequence").str.slice(pl.col("mutation_pos") - 1, 1).alias("seq_aa_at_pos")])

df = df.with_columns([
    pl.when(
        (pl.col("mutation_pos").is_not_null())
        & (pl.col("mutation_pos") >= 1)
        & (pl.col("mutation_pos") <= pl.col("seq_len"))
        & (pl.col("seq_aa_at_pos") == pl.col("mutation_from")),
    ).then(
        # Construct mutated sequence: prefix + new AA + suffix
        pl.col("sequence").str.slice(0, pl.col("mutation_pos") - 1)
        + pl.col("mutation_to")
        + pl.col("sequence").str.slice(pl.col("mutation_pos"), pl.col("seq_len") - pl.col("mutation_pos"))
    ).otherwise(None).alias("mut_sequence")

]).rename({
    "sequence": "wt_sequence",
    "mutations": "mutation"
}).select([
    "wt_sequence",
    "mutation",
    "mut_sequence",
    "solubility_change"
])

# Display summary statistics for solubility change
df.select(pl.col("solubility_change")).describe()

In [ ]:
# Normalize solubility change using sigmoid function (similar to S350 processing)
k_neg = 3.5
k_pos = 0.8
A_neg = 1.0
A_pos = 1.0

sigmoid_expr = (
    pl.when(pl.col("solubility_change") >= 0)
    .then(A_pos * (2 / (1 + (-k_pos * pl.col("solubility_change")).exp()) - 1))
    .otherwise(-A_neg * (2 / (1 + (-k_neg * (-pl.col("solubility_change"))).exp()) - 1))
)

df = df.with_columns(
    sigmoid_expr.alias("target")
)
df

In [ ]:
# Save prepared dataset to Parquet
if not os.path.exists("datasets"):
    os.makedirs("datasets")
df.write_parquet("datasets/ponsol_prepared.parquet")